# Dress-up virtual try-on — Colab notebook

Research implementation of **"Dress-up: deep neural framework for image-based human appearance transfer"**
(Ghodhbani et al., *Multimedia Tools and Applications* 2023).

**Implementation status (Phase 3):**

| Stage | Status |
|---|---|
| A. Pre-treatment (clothing + body segmentation) | implemented: pretrained SCHP substitutes for LIP-JppNet / NBF, DFMM annotations |
| B. Warp network (dual-path U-Net) | implemented and unit-tested (Phase 4); **not yet trained on real data** |
| C. Texture network (alignment + SPADE) | **not implemented yet** (Phase 5) |
| Garment-only (product photo) references | experimental preprocessing only; no compatible warp model |

Every section can be read on its own. Run the cells from top to bottom. Nothing heavy runs automatically. Downloads only happen if you allow them in the settings cell.

## 0. Settings — the only cell you normally edit

In [ ]:
# --- where the code is -------------------------------------------------------------
REPO_URL = ""                       # e.g. "https://github.com/<you>/virtual-tryon.git" (leave "" if you upload a zip)
# Private repo: add a Colab secret named GITHUB_TOKEN (key icon in the left sidebar). Never paste tokens here.
CODE_ZIP_IN_DRIVE = "code/virtual-tryon.zip"   # alternative: zip of the repo stored under DRIVE_PROJECT_ROOT
COLAB_PROJECT_DIR = "/content/virtual-tryon"

# --- storage -------------------------------------------------------------------------
USE_DRIVE = True                    # mount Google Drive for datasets / weights / outputs
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/VirtualTryOn"

# --- what to run ---------------------------------------------------------------------
DRY_RUN = True                      # True: synthetic fixture, CPU is fine, no downloads
CONFIG_FILE = "configs/colab_smoke_test.yaml" if DRY_RUN else "configs/colab_baseline.yaml"
ALLOW_WEIGHT_DOWNLOAD = False       # True: fetch SCHP weights (~255 MiB each) from the authors' Google Drive
EXTRA_OVERRIDES = []                # e.g. ["data.image_size=[256,256]", "data.max_samples=200"]

# --- dataset (DeepFashion-MultiModal, real data only) ----------------------------------
FETCH_DFMM = False                  # extract the archives from Drive to Colab's local disk (needed every session)
DFMM_ALLOW_DOWNLOAD = False         # first session only: download missing archives into Drive (~5.5 GB)
DFMM_PARTS = "images_parsing"       # images + parsing + keypoints; "all" adds DensePose (~5.6 GB more)

# --- Phase 4: warp network ---------------------------------------------------------------
PRECOMPUTE_BODY_MAPS = False        # once: SCHP-Pascal over all 12,701 annotated images (GPU), archived to Drive
RUN_WARP_TRAINING = False           # start/resume warp training (Section 10)
WARP_MAX_STEPS = 2000               # total optimiser steps; re-running resumes from the last checkpoint

## 1. Project overview

**Inputs:** `I_C`, the reference image showing the garment, and `I_B`, the target person. **Output:** `I_G`, the target person wearing the garment.

```
 I_C ──► clothing parser ──► C_seg ─┐
                                    ├─► [B] warp: dual-path U-Net ──► C'_seg ─┐
 I_B ──► body-part parser ──► B_seg ┘        (Phase 4)                       ├─► [C] alignment (corr. + softmax)
                                                                             │       + SPADE generator ──► I_G
 I_C ────────────────────────────────────────────────────────────────────────┘       (Phase 5)
        [A] pre-treatment (this phase)
```

* **C_seg**: the garment layout in the `garment12` label set. Only the classes to be transferred are kept.
* **B_seg**: body parts that do not depend on clothing. `body15` comes from DensePose; `pascal_person_part` comes from SCHP.
* Paper details, deviations and open questions: `docs/paper_analysis.md` and `docs/implementation_deviations.md`.

## 2. Runtime verification

In [ ]:
import os, sys, platform
import torch
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
print("Python :", sys.version.split()[0], "|", platform.platform())
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda, "| in Colab:", IN_COLAB)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()
    print(f"GPU    : {p.name} | {total/2**30:.1f} GB total, {free/2**30:.1f} GB free | device cuda:0")
else:
    print("GPU    : none -> device cpu")
    if not DRY_RUN:
        raise SystemExit("DRY_RUN=False needs a GPU runtime (Runtime > Change runtime type > GPU). "
                         "Set DRY_RUN=True to run the CPU checks.")

## 3. Google Drive setup
Mounting asks you to authorise access in a pop-up. Datasets, weights, checkpoints, caches and outputs live in separate folders under `DRIVE_PROJECT_ROOT` (see `src/utils/paths.py`).

In [ ]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")          # interactive authorisation
    os.makedirs(DRIVE_PROJECT_ROOT, exist_ok=True)
    print("Drive project root:", DRIVE_PROJECT_ROOT)
else:
    print("Drive not mounted (not in Colab or USE_DRIVE=False); using local folders.")

## 4. Code and dependencies
PyTorch is **not** reinstalled. Colab's build already matches its CUDA driver. Only the missing packages from `requirements.txt` are installed.

In [ ]:
import subprocess, zipfile
from pathlib import Path

def locate_project():
    here = Path.cwd()
    for cand in (here, here.parent):
        if (cand / "src" / "config.py").is_file():
            return cand
    if IN_COLAB:
        target = Path(COLAB_PROJECT_DIR)
        if REPO_URL:
            url = REPO_URL
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:  # secret missing or notebook access not granted -> public repo
                token = None
            if token:
                url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)
            if (target / ".git").is_dir():
                cmd = ["git", "-C", str(target), "pull", "--ff-only", url]
            else:
                cmd = ["git", "clone", "--depth", "1", url, str(target)]
            r = subprocess.run(cmd, capture_output=True, text=True)
            msg = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
            print(msg.strip())
            if r.returncode != 0:
                raise RuntimeError("git failed. Private repo? Add the GITHUB_TOKEN secret and allow notebook access.")
            return target
        if (target / "src").is_dir():
            return target
        zip_path = Path(DRIVE_PROJECT_ROOT) / CODE_ZIP_IN_DRIVE
        if zip_path.is_file():
            zipfile.ZipFile(zip_path).extractall(target.parent)
            if (target / "src").is_dir():
                return target
    raise FileNotFoundError("Project code not found. Set REPO_URL, or upload a zip of the repository to "
                            f"{DRIVE_PROJECT_ROOT}/{CODE_ZIP_IN_DRIVE} (top-level folder 'virtual-tryon').")

PROJECT_DIR = locate_project()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("project:", PROJECT_DIR)

In [ ]:
# Version / import verification
import importlib
for mod in ["numpy", "PIL", "cv2", "scipy", "matplotlib", "yaml", "tqdm"]:
    m = importlib.import_module(mod)
    print(f"{mod:10s} {getattr(m, '__version__', '?')}")
from src.utils.device import runtime_report, format_runtime_report
print(format_runtime_report(runtime_report()))

## 5. Configuration
All settings live in `src/config.py` (typed dataclasses with defaults) and in the YAML files in `configs/`.

**Limited GPU memory?** Lower these, roughly in order: `data.image_size` (256 → 128), `training.batch_size`. Then raise `training.grad_accum_steps` to keep the effective batch size. Later phases will add `model.texture_corr_size` (64 → 32; correlation memory grows with the 4th power of this size) and the model widths.

In [ ]:
from src.config import load_config, save_config
from src.utils.reproducibility import seed_everything
from src.utils.paths import ProjectPaths

overrides = list(EXTRA_OVERRIDES)
if IN_COLAB and USE_DRIVE and not DRY_RUN:
    overrides.append(f"paths.drive_root={DRIVE_PROJECT_ROOT}")
if ALLOW_WEIGHT_DOWNLOAD:
    overrides.append("segmentation.allow_weight_download=true")
cfg = load_config(CONFIG_FILE, overrides)
paths = ProjectPaths.from_config(cfg).ensure()
seed_everything(cfg.experiment.seed, cfg.experiment.deterministic)
print("experiment :", cfg.experiment.name, "| mode:", cfg.experiment.run_mode, "| dry run:", cfg.experiment.dry_run)
print("dataset    :", cfg.data.dataset, "at", cfg.data.root)
print("image size :", cfg.data.image_size, "| backends:", cfg.segmentation.clothing_backend, "/", cfg.segmentation.body_backend)
print("storage    :", paths.drive_root.resolve())
save_config(cfg, paths.experiment_dir("outputs", cfg.experiment.name) / "config_resolved.yaml")

## 6a. Dataset acquisition (real data only)
* **Drive** (permanent): zip archives in `<drive>/datasets/DeepFashion-MultiModal/archives/`. They are downloaded once and never extracted inside Drive.
* **Colab local disk** (temporary): extracted to `/content/data/DeepFashion-MultiModal`, rebuilt each session. Training reads from here because it is much faster than the Drive mount.
* **Your PC:** nothing is downloaded.

The first time, set `FETCH_DFMM=True` and `DFMM_ALLOW_DOWNLOAD=True`. After that, `FETCH_DFMM=True` alone is enough.

In [ ]:
from src.data.fetch import fetch_and_stage, storage_plan
if DRY_RUN or not FETCH_DFMM:
    print("Skipped (DRY_RUN or FETCH_DFMM=False). Plan:")
    print(storage_plan(DFMM_PARTS, cfg.data.archive_dir, cfg.data.root))
else:
    fetch_and_stage(cfg, DFMM_PARTS, allow_download=DFMM_ALLOW_DOWNLOAD)

## 6. Data preparation
This builds the index, checks the images and annotations, makes item-disjoint splits (the split file is reused on later runs) and shows samples.

* **DRY_RUN:** a **synthetic** cartoon fixture is generated. It is not real data and only tests the code path.
* **Real data:** download DeepFashion-MultiModal to `<drive>/datasets/DeepFashion-MultiModal` first (see the README). Then run `!python scripts/prepare_dataset.py --config configs/colab_baseline.yaml --inspect` once to confirm the folder names.

In [ ]:
from src.data.dataset import prepare_dataset, PreTreatmentDataset
prepared = prepare_dataset(cfg, progress=False)
if prepared.synthetic:
    print("*** SYNTHETIC FIXTURE - not real data ***")
print(prepared.report.summary())
for split, names in prepared.splits.items():
    s = prepared.stats[split]
    print(f"{split:5s}: {len(names):6d} images | {s['items']:5d} items | {s['items_with_2plus_images']} items with >=2 views")

In [ ]:
import matplotlib.pyplot as plt
from src.data.labels import get_label_set
from src.data.preprocessing import tensor_to_image
from src.inference.visualization import overlay, show_panels

G, B = get_label_set(cfg.labels.garment_label_set), get_label_set(cfg.labels.body_label_set)
from src.data.labels import convert_labels, colorize
from src.data.preprocessing import load_image, load_label_map
recs = prepared.records("train")[:2]
if not all(r.densepose for r in recs):   # "images + parsing" download: no body ground truth yet
    for r in recs:
        img = load_image(prepared.index.root / r.image)
        g = convert_labels(load_label_map(prepared.index.root / r.parsing), "dfmm", G.name)
        show_panels([("image", img), ("garment12 (GT parsing)", overlay(img, g, G, 0.7))], legends=[(G, g)], title=r.name)
train_ds = PreTreatmentDataset(prepared, "train", cfg) if all(r.densepose for r in recs) else []
for i in range(min(2, len(train_ds))):
    s = train_ds[i]
    img = tensor_to_image(s["image"]); g = s["garment"].numpy().astype("uint8"); b = s["body"].numpy().astype("uint8")
    assert img.shape[:2] == g.shape == b.shape, "image/label size mismatch"
    show_panels([("image", img), ("garment12 (GT parsing)", overlay(img, g, G, 0.7)), ("body (DensePose)", overlay(img, b, B, 0.7))],
                legends=[(G, g), (B, b)], title=s["name"])
plt.show()

## 7. Pre-treatment inference (Stage A)
This runs the configured clothing and body backends on two images from the validation split: one is used as the reference and one as the target.

* With the smoke config, the backends read the dataset's own annotations, so no weights are needed.
* With the baseline config, `schp_lip` is used for clothing. It needs the SCHP LIP weights, about 255 MiB. Set `ALLOW_WEIGHT_DOWNLOAD=True`, or put the file in `<drive>/pretrained/schp/` yourself.

In [ ]:
from src.inference.pipeline import PreTreatment, try_on
pre = PreTreatment(cfg)
val = prepared.records("val") or prepared.records("train")
ref_rec = val[0]
tgt_rec = next((r for r in val if r.item_id != ref_rec.item_id), val[-1])
from src.data.preprocessing import load_image
ref_img, tgt_img = load_image(prepared.index.root / ref_rec.image), load_image(prepared.index.root / tgt_rec.image)
res = try_on(ref_img, tgt_img, cfg, reference_key=ref_rec.name, target_key=tgt_rec.name, stop_after="pretreatment",
             pretreatment=pre, save_dir=paths.experiment_dir("outputs", cfg.experiment.name) / "pretreatment_example")
show_panels([("I_C reference", ref_img), ("C_seg", G.palette_array()[res.reference.garment_map]),
             ("I_B target", tgt_img), ("B_seg", B.palette_array()[res.target.body_map])],
            legends=[(G, res.reference.garment_map), (B, res.target.body_map)], title="Stage A outputs")
plt.show()
print("notes:", res.notes or "none")

In [ ]:
# Parser accuracy against DFMM manual parsing (only meaningful for a network backend on real data)
from src.evaluation.evaluate import evaluate_clothing_parser
if pre.clothing.name.startswith("annotation"):
    print("Clothing backend reads the ground truth itself; evaluation skipped.")
else:
    report = evaluate_clothing_parser(pre.clothing, prepared, cfg, split="val", limit=50)
    print(f"{report['backend']}: pixel acc {report['pixel_accuracy']:.3f} | mIoU (no bg) {report['mean_iou']:.3f} on {report['n_images']} val images")
    print({k: (round(v, 3) if v is not None else None) for k, v in report["per_class_iou"].items()})

In [ ]:
# Optional: your own photos (needs network backends, e.g. clothing schp_lip + body schp_pascal with
# labels.body_label_set=pascal_person_part). Annotation backends only work on dataset images.
RUN_UPLOAD = False
if RUN_UPLOAD and IN_COLAB:
    from google.colab import files
    up = files.upload()
    (ref_name, ref_bytes), (tgt_name, tgt_bytes) = list(up.items())[:2]
    import io; from PIL import Image
    ref_u, tgt_u = load_image(Image.open(io.BytesIO(ref_bytes))), load_image(Image.open(io.BytesIO(tgt_bytes)))
    r = try_on(ref_u, tgt_u, cfg, stop_after="pretreatment", pretreatment=pre)
    show_panels([("reference", ref_u), ("C_seg", G.palette_array()[r.reference.garment_map]),
                 ("target", tgt_u), ("B_seg", B.palette_array()[r.target.body_map])],
                legends=[(G, r.reference.garment_map), (B, r.target.body_map)]); plt.show()

In [ ]:
# Experimental: garment-only (product photo) reference. Plain backgrounds only; user supplies the category.
# The result is a FLAT layout that the person-worn warp network cannot use (validated=False).
import numpy as np
from PIL import Image, ImageDraw
demo = Image.new("RGB", (120, 140), (245, 245, 245)); d = ImageDraw.Draw(demo)
d.polygon([(30, 20), (90, 20), (115, 50), (95, 60), (95, 125), (25, 125), (25, 60), (5, 50)], fill=(30, 60, 160))
ref_flat = pre.prepare_reference(np.asarray(demo), mode="garment_only", garment_category="top")
show_panels([("synthetic product photo", np.asarray(demo)), ("flat C_seg", G.palette_array()[ref_flat.garment_map])],
            legends=[(G, ref_flat.garment_map)], title=f"validated={ref_flat.validated}")
plt.show(); print(ref_flat.warnings)

## 8. Warp stage, part 1: body maps for training (SCHP-Pascal, once)
Training needs a B_seg map for every annotated image. The network runs here **once** on the GPU (it does not run inside the data loader). The maps are written to the Colab local disk and zipped into Drive (`cache/body_maps_archives/schp_pascal.zip`). Later sessions only unzip them. This takes roughly tens of minutes on a T4 (not yet measured).

Why SCHP-Pascal maps follow the clothing outline, and what the training does about it: `docs/body_representation.md`.

In [ ]:
from src.data.body_maps import stage_body_maps, precompute_body_maps, archive_body_maps
print("restore from Drive:", stage_body_maps(cfg))
if PRECOMPUTE_BODY_MAPS and not DRY_RUN:
    recs = [r for s in ("train", "val", "test") for r in prepared.records(s)]
    print(precompute_body_maps(cfg, recs, prepared.index.root))   # resumable
    print("archived to", archive_body_maps(cfg))
else:
    print("Precompute skipped (PRECOMPUTE_BODY_MAPS=False or DRY_RUN).")

## 8b. Warp stage, part 2: model checks (dual-path U-Net)
Dual-path U-Net (paper Fig. 8, Eqs. 1-4). This section checks the model before any training:
shapes, parameter count, finite losses on a real batch, and a short overfit test on one batch, where the loss must drop.

In [ ]:
from src.training.train_warp import WarpTrainer, _collate
from src.training.losses import segmentation_ce
trainer = WarpTrainer(cfg, prepared)
g = trainer.gen
print("generator params:", f"{sum(p.numel() for p in g.parameters())/1e6:.1f}M",
      "| discriminator:", f"{sum(p.numel() for p in trainer.disc.parameters())/1e6:.2f}M")
print("feature shapes:", g.feature_shapes())
print(f"train triplets: {len(trainer.train_ds)} ({trainer.train_ds.n_cross_view_available()} with a cross-view partner) | "
      f"val: {len(trainer.val_ds)} | swap pairs: {len(trainer.swap_ds)} | pairing: {cfg.training.warp_pairing}")
batch = _collate([trainer.train_ds[i] for i in range(2)])
print({k: tuple(v.shape) for k, v in batch.items() if hasattr(v, "shape")}, batch["mode"])

In [ ]:
# Overfit smoke test on ONE batch (CE only, a throw-away copy of the model). The loss must fall sharply.
import copy, torch
g_tmp = copy.deepcopy(trainer.gen).train()
opt = torch.optim.Adam(g_tmp.parameters(), lr=1e-3)
c, b, t = batch["cloth"].to(trainer.device), batch["body"].to(trainer.device), batch["target"].to(trainer.device)
for s in range(60):
    loss = segmentation_ce(g_tmp(c, b), t)
    assert torch.isfinite(loss), "non-finite loss"
    if s in (0, 19, 59): print(f"step {s+1}: CE {loss.item():.4f}")
    opt.zero_grad(); loss.backward(); opt.step()
del g_tmp, opt
if torch.cuda.is_available(): print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.2f} GB")

## 9. Texture network
**Not implemented yet. Planned for Phase 5.** Alignment network (Eqs. 5-6, correlation in FP32) and SPADE generator (Eq. 7).

## 10. Training: warp stage
Set `RUN_WARP_TRAINING=True`. Checkpoints go to `<drive>/checkpoints/<experiment>/warp/`: `last.pt` every `training.checkpoint_every` steps, plus `best.pt` (best val mIoU).
**After a disconnect, run the notebook again with the same settings.** Training resumes at the exact batch where it stopped.
To train longer, raise `WARP_MAX_STEPS` and re-run.

What to watch:
* `ce` should fall.
* Validation `mIoU` (garment classes, background excluded) should rise.
* Swap test (garment from one person, body from another): `class_area_agreement` should rise. It measures whether the predicted garment mix (e.g. top + skirt) matches the input; a skirt that turns into trousers scores low. `class_fidelity` is a lenient version of the same check.
* Per-class IoU: rare classes (skirt, outer, dress) lag behind top/pants. Their trend matters more than the mean.

In [ ]:
if RUN_WARP_TRAINING:
    result = trainer.fit(max_steps=WARP_MAX_STEPS)
    print(result)
else:
    print("Skipped (RUN_WARP_TRAINING=False).")

In [ ]:
# Training curves and the latest validation grid (columns: input C_seg | B_seg | prediction | ground truth, grey = not supervised;
# lower rows: swap test, no ground truth)
from PIL import Image
from IPython.display import display
log = trainer.log.read()
tr = [r for r in log if r.get("kind") == "train"]; va = [r for r in log if r.get("kind") == "val"]
if tr:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot([r["step"] for r in tr], [r["ce"] for r in tr]); ax[0].set_title("train CE"); ax[0].set_xlabel("step")
    if va:
        ax[1].plot([r["step"] for r in va], [r["mean_iou"] for r in va], marker="o", label="val mIoU")
        ax[1].plot([r["step"] for r in va], [r["class_fidelity"] for r in va], marker="x", label="swap class fidelity (lenient)")
        ax[1].plot([r["step"] for r in va], [r.get("class_area_agreement", float("nan")) for r in va], marker="s",
                   label="swap area agreement (strict)")
        ax[1].legend(); ax[1].set_xlabel("step")
    plt.show()
if va:
    last = va[-1]
    print(f"latest val (step {last['step']}): mIoU {last['mean_iou']:.3f}")
    print("per-class IoU:", {k: round(v, 3) for k, v in last["per_class_iou"].items() if v is not None and k != "background"})
grids = sorted(trainer.out_dir.glob("val_step*.png"))
if grids:
    print(grids[-1].name); display(Image.open(grids[-1]))
else:
    print("No validation grid yet.")

## 11. End-to-end inference
**Not implemented yet. Planned for Phase 6.** Upload two images and see `I_C`, `I_B`, `C_seg`, `B_seg`, `C'_seg` and `I_G`. Demo mode and trained models will be clearly labelled.

## 12. Evaluation
**Not implemented yet. Planned for Phase 7.** Fixed held-out sets. Mask metrics (mIoU, pixel accuracy) and image metrics (SSIM, LPIPS, FID) are kept separate. Published paper values are recorded only as references.

## 13. Experiments
**Not implemented yet. Planned for Phase 8.** Config-driven experiment runs and a comparison table (see `docs/experiments.md`).

## 14. Saving and exporting
Saved so far: the resolved config, the data validation report, the split file, the pre-treatment example and the warp checkpoint (`last.pt` exists only after training has run). Texture weights and evaluation results come in later phases.

In [ ]:
out_dir = paths.experiment_dir("outputs", cfg.experiment.name)
for p in [out_dir / "config_resolved.yaml", Path(cfg.data.split_file),
          Path(cfg.paths.cache_dir) / "reports" / f"{cfg.data.dataset}_data_report.json", out_dir / "pretreatment_example",
          trainer.ckpt.last]:
    print(("OK      " if p.exists() else "MISSING ") + str(p))
# Reload check: the saved config must reproduce the same settings in a fresh runtime
assert load_config(out_dir / "config_resolved.yaml").to_dict() == cfg.to_dict()
print("saved config reloads identically")